# Drug-Target Interaction Predictor — GPU Training (Colab)

**Steps:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `davis_full.csv` to Google Drive (see Step 1)
3. Run All Cells
4. **Download `best_model.pt` when prompted**
5. Copy it to `models/checkpoints/best_model.pt` in your local repo

In [ ]:
# Step 0: Verify GPU
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu}')
    print(f'VRAM: {vram:.1f} GB')
else:
    raise RuntimeError('No GPU! Runtime -> Change runtime type -> T4 GPU')

In [ ]:
# Step 1: Mount Google Drive and copy dataset
# IMPORTANT: Upload data/raw/davis_full.csv to your Google Drive first!
# (e.g. upload to My Drive/data/raw/davis_full.csv)
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

# Clone repo
!git clone https://github.com/marine9056/drug-target-Predictor.git
%cd drug-target-Predictor

# Try to copy data from Google Drive
drive_data = '/content/drive/MyDrive/data/raw/davis_full.csv'
local_data = 'data/raw/davis_full.csv'
os.makedirs('data/raw', exist_ok=True)

if os.path.exists(drive_data):
    shutil.copy(drive_data, local_data)
    print(f'Copied davis_full.csv from Google Drive ({os.path.getsize(local_data) / 1e6:.1f} MB)')
else:
    print(f'WARNING: {drive_data} not found!')
    print('Upload data/raw/davis_full.csv to your Google Drive first.')
    print('Then re-run this cell.')
    print('\nAvailable files in Drive/MyDrive:')
    for item in os.listdir('/content/drive/MyDrive/'):
        print(f'  {item}')

In [ ]:
# Step 2: Install dependencies
!pip install -q torch torch-geometric rdkit pyyaml requests pandas numpy scipy scikit-learn

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')
print(f'GPU: {torch.cuda.is_available()}')

In [ ]:
# Step 3: Quick data verification
import pandas as pd
if os.path.exists('data/raw/davis_full.csv'):
    df = pd.read_csv('data/raw/davis_full.csv')
    print(f'Dataset: {len(df)} rows, {df.columns.tolist()}')
    print(f'pKd range: {df["kd_value"].min():.2f} - {df["kd_value"].max():.2f}')
    print(f'Unique drugs: {df["drug_smiles"].nunique()}, proteins: {df["protein_sequence"].nunique()}')
else:
    raise FileNotFoundError('davis_full.csv not found! Go back to Step 1.')

In [ ]:
# Step 4: Run GPU training (~15 min on T4)
!python train_gpu.py

In [ ]:
# Step 5: Show results
import os
checkpoint_dir = 'models/checkpoints'
print('Checkpoint files:')
for f in sorted(os.listdir(checkpoint_dir)):
    if f.endswith('.pt'):
        size_mb = os.path.getsize(os.path.join(checkpoint_dir, f)) / 1e6
        print(f'  {f} ({size_mb:.1f} MB)')

if os.path.exists('outputs/gpu_results.txt'):
    print('\n--- Test Set Results ---')
    print(open('outputs/gpu_results.txt').read())

In [ ]:
# Step 6: DOWNLOAD best_model.pt — CRITICAL, session will expire!
from google.colab import files

checkpoint_path = 'models/checkpoints/best_model.pt'
if os.path.exists(checkpoint_path):
    size_mb = os.path.getsize(checkpoint_path) / 1e6
    print(f'Downloading best_model.pt ({size_mb:.1f} MB)...')
    print('Save to: models/checkpoints/best_model.pt in your local repo')
    files.download(checkpoint_path)
else:
    print('ERROR: best_model.pt not found! Training may have failed.')

In [ ]:
# Step 7: Verify checkpoint loads correctly
import yaml
import numpy as np
from src.model import DrugTargetPredictor
from src.data_loader import DavisDataset
from src.featurization import DrugFeaturizer, ProteinFeaturizer
from src.train import DrugTargetDataset, collate_fn
from src.evaluate import calculate_metrics
from torch.utils.data import DataLoader

with open('configs/default.yaml') as f:
    config = yaml.safe_load(f)

dataset = DavisDataset(data_dir=config['paths']['data_dir'])
df = dataset.load(force_download=False)
_, _, test_df = dataset.split_data(df)

drug_feat = DrugFeaturizer(max_atoms=config['data']['max_drug_atoms'])
prot_feat = ProteinFeaturizer(max_length=config['data']['max_protein_length'])
test_dataset = DrugTargetDataset(test_df, drug_feat, prot_feat)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, collate_fn=collate_fn)

model = DrugTargetPredictor(
    drug_encoder_config=config['model']['drug_encoder'],
    protein_encoder_config=config['model']['protein_encoder'],
    fusion_type=config['model']['fusion']['type'],
    fusion_dim=config['model']['fusion']['hidden_dim'],
)
ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        pred = model(batch['drug_data'].to(device), batch['protein'].to(device))
        all_preds.extend(pred.cpu().squeeze().tolist())
        all_labels.extend(batch['target'].squeeze().tolist())

metrics = calculate_metrics(np.array(all_labels), np.array(all_preds))
print('\n--- Checkpoint Verification ---')
for k, v in metrics.items():
    if 'pvalue' not in k:
        print(f'  {k}: {v:.4f}')
print(f'\nEpoch: {ckpt.get("epoch", "unknown")}')
print(f'Val loss: {ckpt.get("best_val_loss", "unknown")}')